# 01 - CaFFe inventory

**CPU runtime.** Measures the dataset; no modelling. Every later choice (tile
size, normalisation, class mapping, splits) rests on these numbers.

Taken from primary sources (details and citations in docs/results.md):

| fact | source |
|---|---|
| zip `download.pangaea.de/dataset/940950/files/data_raw.zip`, CC-BY-4.0, no login | PANGAEA |
| folders `bounding_boxes / fronts / sar_images / zones`, split train / test | PANGAEA |
| name `Glacier_Date_Satellite_ResolutionM_Quality_Orbit(_Modality).png` | PANGAEA |
| test = COL + Mapple (never touched until the final evaluation) | ESSD paper |
| zone values 0 NA, 64 stone, 127 glacier, 254 ocean incl. melange | official code |

Our split: validation = **SI** (all five sensors); training = Crane, DBE, JAC, Jorum.

In [1]:
# --- cell: mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [3]:
# --- cell: data (zip on Drive, unpacked to local disk) ---
from pathlib import Path
from sartransfer.data import inventory as inv

DRIVE = Path("/content/drive/MyDrive")
assert DRIVE.exists(), "mount Drive first (cell 1)"
PROJECT = DRIVE / "sar-transfer"
RAW, RES = PROJECT / "raw", PROJECT / "results"
RES.mkdir(parents=True, exist_ok=True)
CAFFE = Path("/content/caffe")      # local disk: thousands of small PNGs are slow over Drive

inv.unzip_caffe(inv.fetch_caffe_zip(RAW), CAFFE)   # 2.86 GB download only the first time
DATA = inv.find_data_root(CAFFE)
print("data root:", DATA)

already present: /content/drive/MyDrive/sar-transfer/raw/data_raw.zip  (2.86 GB)
data root: /content/caffe/data_raw


In [4]:
# --- cell: folder tree ---
inv.tree(DATA, max_depth=2, sample_files=2)

data_raw/  [0 files]
  bounding_boxes/  [681 files]
      COL_2011-06-18_TDX_7_1_024_front_extent_coord.txt
      COL_2011-07-21_TDX_7_1_024_front_extent_coord.txt
      ... (679 more)
  fronts/  [0 files]
    test/  [122 files]
        COL_2011-06-18_TDX_7_1_024_front.png
        COL_2011-07-21_TDX_7_1_024_front.png
        ... (120 more)
    train/  [559 files]
        Crane_2002-11-09_ERS_20_2_061_front.png
        Crane_2002-11-18_ERS_20_3_195_front.png
        ... (557 more)
  sar_images/  [0 files]
    test/  [122 files]
        COL_2011-06-18_TDX_7_1_024.png
        COL_2011-07-21_TDX_7_1_024.png
        ... (120 more)
    train/  [559 files]
        Crane_2002-11-09_ERS_20_2_061.png
        Crane_2002-11-18_ERS_20_3_195.png
        ... (557 more)
  zones/  [0 files]
    test/  [122 files]
        COL_2011-06-18_TDX_7_1_024_zones.png
        COL_2011-07-21_TDX_7_1_024_zones.png
        ... (120 more)
    train/  [559 files]
        Crane_2002-11-09_ERS_20_2_061_zones.png
       

In [5]:
# --- cell: scan every file and count ---
import pandas as pd
pd.set_option("display.width", 160, "display.max_rows", 120)

df = inv.scan(DATA)                        # warns if a name does not fit the scheme
print(f"{len(df)} files\n")
c = inv.counts(df, kind="sar_images")
for name in ("per_glacier_split", "per_satellite_split", "per_band_split",
             "per_glacier_satellite", "resolution_m", "quality"):
    print(f"\n===== {name} =====")
    print(c[name])

2043 files


===== per_glacier_split =====
split    test  train  All
glacier                  
COL        65      0   65
Crane       0     69   69
DBE         0    133  133
JAC         0    159  159
Jorum       0     77   77
Mapple     57      0   57
SI          0    121  121
All       122    559  681

===== per_satellite_split =====
split      test  train  All
satellite                  
ENVISAT      10     72   82
ERS           2     52   54
PALSAR        8     40   48
RSAT          0     54   54
S1           33     15   48
TDX          47      0   47
TSX          22    326  348
All         122    559  681

===== per_band_split =====
split  test  train  All
band                   
C        45    193  238
L         8     40   48
X        69    326  395
All     122    559  681

===== per_glacier_satellite =====
satellite  ENVISAT  ERS  PALSAR  RSAT  S1  TDX  TSX  All
glacier                                                 
COL              0    0       0     0  18   47    0   65
Crane 

In [6]:
# --- cell: 10 parsed file names ---
cols = ["glacier", "date", "satellite", "band", "resolution_m", "quality", "orbit"]
print(df[df.kind == "sar_images"][["file"] + cols].head(10).to_string(index=False))

                          file glacier       date satellite band  resolution_m  quality orbit
COL_2011-06-18_TDX_7_1_024.png     COL 2011-06-18       TDX    X           7.0        1   024
COL_2011-07-21_TDX_7_1_024.png     COL 2011-07-21       TDX    X           7.0        1   024
COL_2011-08-12_TDX_7_1_024.png     COL 2011-08-12       TDX    X           7.0        1   024
COL_2011-08-23_TDX_7_1_024.png     COL 2011-08-23       TDX    X           7.0        1   024
COL_2011-09-03_TDX_7_1_024.png     COL 2011-09-03       TDX    X           7.0        1   024
COL_2011-10-11_TDX_7_1_092.png     COL 2011-10-11       TDX    X           7.0        1   092
COL_2011-11-13_TDX_7_1_092.png     COL 2011-11-13       TDX    X           7.0        1   092
COL_2011-12-11_TDX_7_1_024.png     COL 2011-12-11       TDX    X           7.0        1   024
COL_2012-01-02_TDX_7_1_024.png     COL 2012-01-02       TDX    X           7.0        1   024
COL_2012-02-04_TDX_7_1_024.png     COL 2012-02-04       TDX 

In [7]:
# --- cell: image size, dtype, bit depth (all 681 images, a few minutes) ---
sar = df[df.kind == "sar_images"]
stats = inv.image_stats(sar["path"])
print(stats[["pil_mode", "dtype", "bits_per_pixel"]].value_counts().to_frame("n"))
print("\nvalue range:", stats["min"].min(), "..", stats["max"].max())
print("height", stats.height.min(), "-", stats.height.max(),
      "| width", stats.width.min(), "-", stats.width.max())
print("\n=== distinct image shapes per glacier ===")
print(inv.dimension_consistency(stats, df))

                                 n
pil_mode dtype bits_per_pixel     
L        uint8 8               681

value range: 0 .. 255
height 405 - 3561 | width 382 - 4476

=== distinct image shapes per glacier ===
         n_images  n_distinct_shapes  min_h  max_h  min_w  max_w
glacier                                                         
COL            65                  2    702   2107   1492   4476
Crane          69                  3   1187   3561    922   2768
DBE           133                  4    939   2819   1027   3081
JAC           159                  1   3143   3143   2639   2639
Jorum          77                  3    651   1955    998   2996
Mapple         57                  3    405   1215    382   1146
SI            121                  4    943   2828   1145   3435


In [8]:
# --- cell: label values, sampled across ALL glaciers (10 files each) ---
for kind in ("zones", "fronts"):
    paths = df[df.kind == kind].groupby("glacier").head(10)["path"]
    print(f"===== {kind} ({len(paths)} files) =====")
    print(inv.unique_values(paths).to_string(index=False), "\n")
print("expected zones: 0 NA, 64 stone, 127 glacier, 254 ocean | fronts: 0 / 255")

===== zones (70 files) =====
70 files, 4 distinct values
 value    pixels  share_%
     0  14764027    6.754
    64  45668438   20.890
   127 129239819   59.118
   254  28939519   13.238 

===== fronts (70 files) =====
70 files, 2 distinct values
 value    pixels  share_%
     0 218526434   99.961
   255     85369    0.039 

expected zones: 0 NA, 64 stone, 127 glacier, 254 ocean | fronts: 0 / 255


In [9]:
# --- cell: bounding boxes (used later: the official MDE masks predictions outside them) ---
bb = inv.scan_bboxes(DATA)
f0 = sorted((DATA / "bounding_boxes").glob("*.txt"))[0]
print(f0.name); print(f0.read_text().strip())
print("\n", len(bb), "boxes | pixel coordinates of each image; the official "
      "validate_or_test.py zeroes predicted front pixels outside the box before scoring")

COL_2011-06-18_TDX_7_1_024_front_extent_coord.txt
x,y
270.743801579289,1959.30115735809
270.743801579289,909.268400764701
2634.34895854926,909.268400764701
2634.34895854926,1959.30115735809

 681 boxes | pixel coordinates of each image; the official validate_or_test.py zeroes predicted front pixels outside the box before scoring


In [10]:
# --- cell: size on disk + save the inventory to Drive ---
for folder in sorted(p for p in DATA.iterdir() if p.is_dir()):
    print(f"{folder.name:<16} {inv.dir_size_bytes(folder)/1e9:6.2f} GB")
df.drop(columns=["path"]).to_csv(RES / "caffe_inventory.csv", index=False)
stats.to_csv(RES / "caffe_image_stats.csv", index=False)
print("\nwrote caffe_inventory.csv and caffe_image_stats.csv to", RES)

bounding_boxes     0.00 GB
fronts             0.01 GB
sar_images         2.83 GB
zones              0.03 GB

wrote caffe_inventory.csv and caffe_image_stats.csv to /content/drive/MyDrive/sar-transfer/results
